# E1 — gpt-oss-20b 도메인 사전학습 (경제 문장 50만, Colab) · 팀원 실행용

**무엇을 하나**: OpenAI 공개 모델 gpt-oss-20b에 경제 기사 문장 50만 개를 더 읽혀(다음 단어 맞히기) 금융 문체에 익숙하게 만든다. 본체(20B)는 고정하고 **LoRA(작은 덧붙임 가중치)만 학습**한다. 라벨(span)은 쓰지 않는다. 이후 이 모델로 '문장 → 개체 목록을 글로 답하는' 방식의 금융 NER 파인튜닝을 한다.

## 런타임 설정 (런타임 → 런타임 유형 변경)
| 항목 | 고를 것 |
|---|---|
| GPU | **H100** 또는 **G4(RTX PRO 6000, 96GB)** 또는 **A100 80GB** — GPU 메모리 **70GB 이상** 필수 |
| 안 되는 GPU | A100 40GB, L4, T4 (본체를 풀면 약 42GB라 안 올라감) — 1번 셀이 자동으로 막음 |
| RAM | 표준 (고용량 RAM 체크 불필요) |

## 예상 자원 (⚠ 추정 — 4번 셀이 실제 속도를 재서 다시 출력함)
| 항목 | 값 |
|---|---|
| 데이터 | 50만 문장 = 약 2,075만 토큰(문장당 41.5토큰, 실측) → 512토큰 블록 약 4만 개 → **약 1,250스텝** |
| GPU 메모리 | 약 50~65GB 예상 (본체 bf16 약 42GB + LoRA·활성값) |
| 시간 | 준비(모델 13GB 다운로드·토큰화) 15~20분 + 학습 **약 1~3시간** (추정) |
| 컴퓨팅 단위 | 약 10~25 CU (추정. A100 80GB 시간당 약 7.5 CU 기준) |
| 드라이브 사용 | 토큰 캐시 약 80MB + 체크포인트·결과 LoRA 수백 MB |

## 준비 (1회) — 각자 내 드라이브에서 작업
1. 공유 폴더 링크(노션 페이지)에서 파일 3개를 **내려받는다**: `E1_gptoss_도메인사전학습_Colab_팀원용.ipynb`, `dapt_gptoss_colab.py`, `econ_dapt.txt.gz`(295MB)
2. **내 드라이브**에 `gptoss_dapt` 폴더를 만들고 세 파일을 그 안에 **올린다**(업로드가 끝날 때까지 기다리기 — 2번 셀이 말뭉치 크기 309,309,742바이트로 확인)
3. 그 폴더의 노트북을 더블클릭해 Colab으로 연다. 2번 셀은 `/content/drive/MyDrive/gptoss_dapt` 를 쓰고, 없으면 내 드라이브 전체에서 찾는다. 다른 이름으로 만들었으면 2번 셀 `ROOT` 만 고친다
4. **학습 결과는 Colab 로컬 디스크(`/content/out`)에 저장**하고, 체크포인트는 내 드라이브 `gptoss_dapt/ckpt/` 에도 복사한다(끊겨도 이어서 학습). 끝나면 6번 셀이 LoRA·결과 json만 내 드라이브 `gptoss_dapt/결과_dapt_gptoss_econ500k/` 에 올리고, 7번 셀이 압축 파일을 내 PC로 내려받은 뒤 런타임을 반납한다. 20B 본체(13GB)도 Colab 로컬(`/content/hf`)에 받으므로 내 드라이브 용량은 수백 MB면 충분

## 실행
- **위에서부터 차례로 실행.** 4번(속도 측정, 50스텝)이 끝나면 출력된 `[속도]` 줄을 팀 채널에 공유하고 5번(본 학습)으로 넘어간다.
- 체크포인트는 Colab 로컬과 **내 드라이브 `gptoss_dapt/ckpt/` 양쪽에 저장**된다(100스텝마다, 드라이브엔 최근 2개). 연결이 끊기면 **1~3번과 5번을 다시 실행** → 드라이브의 마지막 체크포인트를 가져와 이어서 학습.
- 메모리 부족(CUDA out of memory)이 나면 5번 셀의 `MB, GA = 4, 8` 을 `2, 16` 으로 바꿔 다시 실행(학습량은 같음).
- 7번 셀은 **6번에서 저장 확인·드라이브 업로드가 통과했을 때만** 내려받기 후 런타임을 반납한다.
- 끝나면 내 드라이브 `gptoss_dapt/결과_dapt_gptoss_econ500k/` 폴더(또는 내려받은 zip)를 공유 폴더에 올려 주세요.

## 데이터 주의
- 이 말뭉치는 **금융 골드셋 510문장과 Claude 라벨 표본 12,510문장을 뺀** 경제 문장이다(평가 문장이 사전학습에 섞이지 않게). 다른 문장을 추가하지 말 것.
- API 키는 필요 없다(Hugging Face 공개 모델). 토큰을 노트북에 붙여 넣지 말 것.


In [ ]:
# 1. GPU 확인 (70GB 이상이어야 함)
import torch, subprocess
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (런타임 → 런타임 유형 변경)'
p = torch.cuda.get_device_properties(0)
print(p.name, f'{p.total_memory/2**30:.1f}GB', 'bf16', torch.cuda.is_bf16_supported())
assert p.total_memory > 70 * 2**30, 'GPU 메모리가 70GB 미만입니다 → H100 / G4 / A100 80GB 로 바꾸세요'
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# 2. 드라이브 연결과 파일 확인 (내 드라이브의 gptoss_dapt 폴더)
from google.colab import drive
drive.mount('/content/drive')
import os, glob
ROOT = '/content/drive/MyDrive/gptoss_dapt'   # 폴더 이름·위치를 다르게 만들었으면 여기만 수정
if not os.path.exists(f'{ROOT}/dapt_gptoss_colab.py'):
    found = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/dapt_gptoss_colab.py', recursive=True)]
    print('기본 위치에 없음 → 내 드라이브에서 찾은 위치:', found)
    assert found, '내 드라이브에 dapt_gptoss_colab.py 가 없습니다 → 준비 1~2번(내 드라이브에 gptoss_dapt 폴더 만들고 파일 3개 올리기)을 확인하세요'
    ROOT = found[0]
print('ROOT =', ROOT)
for f in ['econ_dapt.txt.gz', 'dapt_gptoss_colab.py']:
    path = f'{ROOT}/{f}'
    print(f, os.path.exists(path), f'{os.path.getsize(path)/2**20:.0f}MB' if os.path.exists(path) else '← 없음')
assert os.path.getsize(f'{ROOT}/econ_dapt.txt.gz') == 309_309_742, '말뭉치 파일이 덜 올라왔거나 다른 파일입니다(업로드가 끝났는지 확인)'
print('파일 확인 OK')

In [ ]:
# 3. 라이브러리 (gpt-oss 는 transformers 4.55 이상 필요)
!pip install -q -U transformers peft accelerate
import transformers, peft
print('transformers', transformers.__version__, '· peft', peft.__version__)
os.environ['HF_HOME'] = '/content/hf'   # 모델 13GB 는 Colab 디스크에 받음(드라이브 아님)

In [ ]:
# 4. 속도 측정 (50스텝, 저장 안 함) — 처음엔 모델 다운로드·토큰화로 15~20분 걸림
NAME = 'dapt_gptoss_econ500k'
OUT = '/content/out'   # 학습 결과는 Colab 로컬 디스크에 저장 → 끝나면 6번 셀이 드라이브에 올림
!cd /content && HF_HOME=/content/hf python "{ROOT}/dapt_gptoss_colab.py" --corpus "{ROOT}/econ_dapt.txt.gz" --out_root {OUT} --name {NAME} --lines 500000 --max_steps 50 --probe 1
# 출력의 [속도] 줄(초/스텝, 토큰/초, 전체 예상 시간, GPU 최고 메모리)을 팀 채널에 공유

In [ ]:
# 5. 본 학습 — 체크포인트는 Colab 로컬(/content/out)에 저장하고, 저장할 때마다 내 드라이브(ROOT/ckpt)에도 복사
#    끊기면 1~3번과 이 셀을 다시 실행 → 로컬에 없으면 드라이브의 마지막 체크포인트를 가져와 이어서 학습
MB, GA = 4, 8      # 메모리 부족이면 2, 16
!cd /content && HF_HOME=/content/hf python "{ROOT}/dapt_gptoss_colab.py" --corpus "{ROOT}/econ_dapt.txt.gz" --out_root {OUT} --mirror "{ROOT}" --name {NAME} --lines 500000 --micro_batch {MB} --grad_accum {GA}

In [ ]:
# 6. 결과 확인 → 결과만 드라이브(ROOT/결과)에 올리고 압축 파일 만들기
import json, shutil, os, filecmp
r = json.load(open(f'{OUT}/results/{NAME}.json'))
print('저장 확인', r.get('verify_ok'), '· 스텝', r.get('steps'), '· 학습', r.get('train_minutes'), '분')
print('검증 손실  학습 전', round(r['eval_loss_before'], 4), '→ 학습 후', round(r.get('eval_loss_after', float('nan')), 4))
UP = f'{ROOT}/결과_{NAME}'
shutil.copytree(f'{OUT}/models/{NAME}', f'{UP}/models/{NAME}', dirs_exist_ok=True)
os.makedirs(f'{UP}/results', exist_ok=True); shutil.copy(f'{OUT}/results/{NAME}.json', f'{UP}/results/')
same = all(os.path.getsize(f'{OUT}/models/{NAME}/{f}') == os.path.getsize(f'{UP}/models/{NAME}/{f}') for f in os.listdir(f'{OUT}/models/{NAME}'))
print('드라이브 업로드', UP, '· 크기 일치', same)
shutil.make_archive(f'/content/{NAME}', 'zip', UP); print('압축:', f'/content/{NAME}.zip', os.path.getsize(f'/content/{NAME}.zip') // 2**20, 'MB')
VERIFY_OK = bool(r.get('verify_ok')) and same

In [ ]:
# 7. 내 PC로 내려받기(선택) + 런타임 반납 (저장 확인·드라이브 업로드가 통과했을 때만)
import time
from google.colab import files, runtime
if VERIFY_OK:
    files.download(f'/content/{NAME}.zip')
    print('내려받기 시작 — 2분 뒤 런타임 반납'); time.sleep(120)
    runtime.unassign()
else:
    print('저장 확인 또는 드라이브 업로드 실패 → 반납하지 않음. 출력을 공유해 주세요')